# 75 — Space Alignment Problem (arahan dosen #2)

**Tujuan:** evaluasi apakah feature space image (CNN TL 256-d) dan landmark (FCNN 128-d) **sejajar** di latent space.

**Motivasi:** kalau space tidak aligned, Intermediate Fusion (concat) suboptimal → justifikasi kenapa Late Fusion (decision-level) mungkin lebih robust di natural data. Post val-tuning fix, Intermediate TL justru juara (0.521 > Late Fusion TL 0.466) — perlu dicek apakah alignment memang kuat, atau Intermediate menang karena alasan lain (augmentation effect di B3, joint training dynamics).

**Metode (sesuai spec eksplorasi_lanjutan.md #2):**
1. **CCA** (Canonical Correlation Analysis) — linear alignment top-k components
2. **t-SNE** joint embedding — cek apakah same-class cluster di kedua stream
3. **Per-class cosine similarity** — paired (image_i, landmark_i) aligned per-sample
4. **Cross-modal retrieval** — given image feature, find matching landmark di test set (top-k accuracy)

**Checkpoint:**
- CNN TL 4c B1 — `models/frontonly_conf60/4class_tl/cnn_tl_b1.pth` (256-d features via `extract_features`)
- FCNN 4c B2 — `models/frontonly_conf60/4class/fcnn_b2.pth` (128-d features via `extract_features`)

**Output:**
- `docs/figures/space_alignment/cca_correlations.png`
- `docs/figures/space_alignment/tsne_per_stream.png`
- `docs/figures/space_alignment/retrieval_topk.png`
- `models/frontonly_conf60/space_alignment/alignment_metrics.json` — quantitative scores

**Estimasi:** ~30 menit (analytical only, no training)

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import CCA
from sklearn.manifold import TSNE

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionCNNTransfer, EmotionFCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DATA_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
CKPT_DIR = PROJECT_ROOT / 'models' / 'frontonly_conf60'
OUT_FIG  = PROJECT_ROOT / 'docs' / 'figures' / 'space_alignment'
OUT_JSON = PROJECT_ROOT / 'models' / 'frontonly_conf60' / 'space_alignment'
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_JSON.mkdir(parents=True, exist_ok=True)

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)
EMOTIONS = ['neutral', 'happy', 'sad', 'negative']
COLORS = ['#4A6A8A', '#5A8055', '#A87143', '#9E5070']

In [ ]:
# ── Load test data + checkpoints ──
t_img = np.load(DATA_DIR / 'X_test_images.npy').astype(np.float32)
t_lm  = np.load(DATA_DIR / 'X_test_landmarks.npy').astype(np.float32)
t_y   = REMAP_4[np.load(DATA_DIR / 'y_test.npy')]
print(f'Test: {len(t_y)} samples, class dist: {np.bincount(t_y, minlength=4).tolist()}')

cnn = EmotionCNNTransfer(num_classes=4).to(device)
cnn_ckpt = CKPT_DIR / '4class_tl' / 'cnn_tl_b1.pth'
cnn.load_state_dict(torch.load(cnn_ckpt, map_location=device, weights_only=True))
cnn.eval()
print(f'CNN TL loaded: {cnn_ckpt}')

fcnn = EmotionFCNN(num_classes=4).to(device)
fcnn_ckpt = CKPT_DIR / '4class' / 'fcnn_b2.pth'
fcnn.load_state_dict(torch.load(fcnn_ckpt, map_location=device, weights_only=True))
fcnn.eval()
print(f'FCNN loaded: {fcnn_ckpt}')

In [ ]:
# ── Extract features (256-d CNN, 128-d FCNN) ──
img_t = torch.from_numpy(t_img).permute(0, 3, 1, 2)
lm_t  = torch.from_numpy(t_lm)

feat_cnn, feat_fcnn = [], []
with torch.no_grad():
    for i in range(0, len(img_t), 64):
        f_cnn  = cnn.extract_features(img_t[i:i+64].to(device))
        f_fcnn = fcnn.extract_features(lm_t[i:i+64].to(device))
        feat_cnn.append(f_cnn.cpu().numpy())
        feat_fcnn.append(f_fcnn.cpu().numpy())
feat_cnn  = np.concatenate(feat_cnn)
feat_fcnn = np.concatenate(feat_fcnn)
print(f'CNN feat: {feat_cnn.shape}   FCNN feat: {feat_fcnn.shape}')

## (1) CCA — Canonical Correlation Analysis

In [ ]:
n_comp = min(20, feat_cnn.shape[1], feat_fcnn.shape[1])
cca = CCA(n_components=n_comp)
cca.fit(feat_cnn, feat_fcnn)
cnn_c, fcnn_c = cca.transform(feat_cnn, feat_fcnn)

corrs = np.array([np.corrcoef(cnn_c[:, i], fcnn_c[:, i])[0, 1] for i in range(n_comp)])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(1, n_comp + 1), corrs, color='#4A6A8A', edgecolor='black', linewidth=0.5)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='0.5 (moderate alignment)')
ax.set_xlabel('Canonical component'); ax.set_ylabel('Correlation')
ax.set_title(f'CCA — CNN (256-d) vs FCNN (128-d)   |   top-5 mean = {corrs[:5].mean():.3f}')
ax.set_xticks(range(1, n_comp + 1))
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_FIG / 'cca_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'CCA top-5 correlations: {[f"{c:.3f}" for c in corrs[:5]]}')
print(f'CCA top-5 mean: {corrs[:5].mean():.3f}  |  top-10 mean: {corrs[:10].mean():.3f}')

## (2) t-SNE Joint Embedding

In [ ]:
# Project CNN 256→128 by truncation for joint 2D embed (same dim)
combined = np.vstack([feat_cnn[:, :128], feat_fcnn])
labels = np.concatenate([t_y, t_y])
origin = np.array([0]*len(t_y) + [1]*len(t_y))  # 0=CNN, 1=FCNN

tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
emb = tsne.fit_transform(combined)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for cls in range(4):
    m_cnn = (labels == cls) & (origin == 0)
    m_lm  = (labels == cls) & (origin == 1)
    axes[0].scatter(emb[m_cnn, 0], emb[m_cnn, 1], c=COLORS[cls], label=EMOTIONS[cls], s=10, alpha=0.65)
    axes[1].scatter(emb[m_lm,  0], emb[m_lm,  1], c=COLORS[cls], label=EMOTIONS[cls], s=10, alpha=0.65)
axes[0].set_title('CNN features (image stream)'); axes[0].legend(fontsize=8)
axes[1].set_title('FCNN features (landmark stream)'); axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(OUT_FIG / 'tsne_per_stream.png', dpi=150, bbox_inches='tight')
plt.show()

## (3) Per-Class Cosine Similarity (CCA-aligned)

In [ ]:
def normalize(x, eps=1e-9):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

a = normalize(cnn_c)
b = normalize(fcnn_c)
diag_cos = (a * b).sum(axis=1)  # per-sample cosine between paired (image_i, landmark_i)

per_class = {}
print('Per-class paired cosine similarity (CCA-aligned):')
for cls in range(4):
    mask = t_y == cls
    if mask.sum() == 0:
        per_class[EMOTIONS[cls]] = None
        continue
    mean_cos = float(diag_cos[mask].mean())
    std_cos  = float(diag_cos[mask].std())
    per_class[EMOTIONS[cls]] = {'n': int(mask.sum()), 'mean': mean_cos, 'std': std_cos}
    print(f'  {EMOTIONS[cls]:>10}: cos = {mean_cos:.3f} ± {std_cos:.3f}  (n={mask.sum()})')

overall_cos = float(diag_cos.mean())
print(f'\n  overall: cos = {overall_cos:.3f}')

## (4) Cross-Modal Retrieval — Given Image Feature, Find Matching Landmark

Asumsi alignment kuat: query image feature → paired landmark feature harus di top-k nearest neighbors. Metrik: top-1, top-5, top-10 retrieval accuracy di test set (929 samples).

In [ ]:
# Retrieval di CCA-aligned space (dimension matched)
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(a, b)  # (N, N) — image_i ↔ landmark_j
ranks = np.argsort(-sim_matrix, axis=1)  # descending, each row = ranked landmark indices

# Top-k accuracy: how often paired landmark_i is in top-k neighbors of image_i
def topk_accuracy(ranks, k):
    n = len(ranks)
    hits = sum(1 for i in range(n) if i in ranks[i, :k])
    return hits / n

topk_results = {}
for k in [1, 5, 10, 20, 50]:
    acc = topk_accuracy(ranks, k)
    topk_results[f'top_{k}'] = acc
    print(f'  top-{k:>3} retrieval accuracy: {acc:.4f}')

# Random baseline
print(f'\n  Random baseline top-5 ≈ 5/{len(ranks)} = {5/len(ranks):.4f}')
print(f'  Random baseline top-10 ≈ 10/{len(ranks)} = {10/len(ranks):.4f}')

# Viz
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ks = list(topk_results.keys())
vals = list(topk_results.values())
ax.bar(ks, vals, color='#5A8055', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Retrieval accuracy'); ax.set_xlabel('top-k')
ax.set_title(f'Cross-modal retrieval — CNN image → paired FCNN landmark (n={len(ranks)})')
ax.set_ylim(0, 1)
for k, v in zip(ks, vals):
    ax.text(k, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_FIG / 'retrieval_topk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save quantitative metrics ──
metrics = {
    'cca_top_5_mean':  float(corrs[:5].mean()),
    'cca_top_10_mean': float(corrs[:10].mean()),
    'cca_correlations': corrs.tolist(),
    'paired_cosine_overall': overall_cos,
    'paired_cosine_per_class': per_class,
    'cross_modal_retrieval': topk_results,
    'test_n_samples': int(len(t_y)),
    'checkpoints': {
        'cnn': str(cnn_ckpt.relative_to(PROJECT_ROOT)),
        'fcnn': str(fcnn_ckpt.relative_to(PROJECT_ROOT)),
    },
}
out_path = OUT_JSON / 'alignment_metrics.json'
with open(out_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved: {out_path}')

## Interpretasi

| Indikator | Nilai | Arti |
|---|---|---|
| CCA top-5 mean > 0.5 | strong linear alignment | Intermediate Fusion (concat) justified |
| CCA top-5 mean 0.3-0.5 | moderate | kedua fusion strategy viable |
| CCA top-5 mean < 0.3 | weak | Late Fusion (decision-level) lebih robust |
| t-SNE same-class cluster di kedua stream | semantic alignment ok | kedua stream capture emotion-relevant features |
| Per-class cosine tinggi | per-sample pairing kuat | feature-level fusion berfungsi baik |
| Cross-modal retrieval top-5 > 0.3 | aligned enough untuk retrieval | strong argument alignment |
| Cross-modal top-5 ~ random | essentially independent | streams memang complementary, bukan redundant |

**Output masuk ke BAB Discussion**: justifikasi mengapa Intermediate TL juara 4-class (0.521) vs Late Fusion TL (0.466) — mungkin karena alignment cukup kuat sehingga concat fusion bisa extract informasi dari kedua stream secara efektif.